In [ ]:
import logging
import random
import pandas as pd
import random
import matplotlib.pylab as plt
import numpy as np
import os
import sys
from pathlib import Path
logger = logging.getLogger(name=__name__)

repo_root = Path.cwd()
while not (repo_root / "src").exists() and repo_root != repo_root.parent:
    repo_root = repo_root.parent
sys.path.append(str(repo_root))


### importing my own notebooks
import src.vertexModel2 as vertexModel2
import src.inputMechanicalParametersModel2 as MechanicalParams2
import src.auxFunctions as auxFunctions
import src.auxFunctionsHomeostasis as auxFunctionsHomeostasis


import warnings
warnings.filterwarnings('ignore')


In [ ]:
def homeostasis_simulation(
    output_folder,
    max_cycles,
    max_layers,
    distance,
    collapse_edge_mode: str = "random",
    divide_vertex_mode: str = "max",
):
    """
    Execute a homeostasis simulation alternating edge collapse and vertex division.

    Options for choosing edge for collapse (apoptosis):

    "min" - collapsing shortest edge in the model
    "max" - collapsing longest edge in the model
    "random" - collapsing a random edge

    Options for choosing vertex for division:

    "min" - dividing vertex connected to smallest total length of attached edges 
    "max" -  dividing vertex connected to largest total length of attached edges 
    "random" - dividing a random vertex

    """
    # Creating output directory if it does not exist
    if not os.path.exists(output_folder):
        os.makedirs(output_folder)

    # Initialising tissue and mechanical parameters
    cellmap, geom, energyContributions_model = vertexModel2.initialize(40)
    cellmap = MechanicalParams2.update(cellmap)

    # Fixing vertices at boundary 
    boundary_edges, boundary_faces, inside_edges, outside_edges, inside_faces, inside_vertices, outside_vertices = auxFunctions.identify_boundary_layers(cellmap, 1)
    high_viscosity_value = 1000000  
    for vertex_id in outside_vertices:
        if vertex_id in cellmap.vert_df.index:
            cellmap.vert_df.at[vertex_id, "viscosity"] = high_viscosity_value

    # Allowing model to relax
    energyContributions_model.compute_energy(cellmap)
    cellmap, geom, _, _, _ = vertexModel2.solveEuler(
        cellmap, geom, energyContributions_model, endTime=100
    )

    # Saving initial model state
    fig, axes = auxFunctions.view(cellmap, geom)
    initial_xlim = axes.get_xlim()
    initial_ylim = axes.get_ylim()
    fig.savefig(os.path.join(output_folder, "cycle_0_initial_frame.png"), dpi=150)
    plt.close(fig)
    auxFunctions.save_simulation_state(cellmap, output_folder, filename="cycle_0_initial_state.pkl")

    # Main simulation loop
    for cycle_index in range(max_cycles):
        print(f"Processing cycle {cycle_index + 1} of {max_cycles}")

        # Identifying tissue boundary 
        boundary_edges, boundary_faces, inside_edges, outside_edges, inside_faces, inside_vertices, outside_vertices = (
            auxFunctions.identify_boundary_layers(cellmap, max_layers)
        )

        if not inside_edges or not inside_vertices:
            print("No interior edges or vertices available. Terminating simulation.")
            break

        # Edge collapse
        if collapse_edge_mode == "random":
            chosen_edge = random.choice(inside_edges)
        elif collapse_edge_mode == "min":
            edge_lengths = cellmap.edge_df.loc[inside_edges, "length"]
            chosen_edge = edge_lengths.idxmin()
        elif collapse_edge_mode == "max":
            edge_lengths = cellmap.edge_df.loc[inside_edges, "length"]
            chosen_edge = edge_lengths.idxmax()
        else:
            raise ValueError("collapse_edge_mode must be 'random', 'min', or 'max'")

        print(f"Collapsing edge {chosen_edge} (mode: {collapse_edge_mode})")

        # Executing edge collapse
        cellmap, new_vertex_from_collapse = auxFunctionsHomeostasis.collapse_single_edge_homeostasis(cellmap, geom, chosen_edge)

        # Saving post-collapse visualisation
        fig, axes = auxFunctions.highlight_vertices(
            cellmap, geom, [new_vertex_from_collapse],
            default_color="white", highlight_color="red",
            default_size=5, highlight_size=100
        )
        axes.set_xlim(initial_xlim)
        axes.set_ylim(initial_ylim)

        fig.savefig(
            os.path.join(output_folder, f"cycle_{cycle_index + 1}_collapse_frame.png"),
            dpi=150
        )
        plt.close(fig)
        
        # Allowing model to relax after edge collapse
        energyContributions_model.compute_energy(cellmap)
        cellmap, geom, _, _, _ = vertexModel2.solveEuler(
            cellmap, geom, energyContributions_model, endTime=40
        )

        # Re-identifying tissue boundary 
        boundary_edges, boundary_faces, inside_edges, outside_edges, inside_faces, inside_vertices, outside_vertices = (
            auxFunctions.identify_boundary_layers(cellmap, max_layers)
        )

        if not inside_vertices:
            print("No interior vertices available for division. Terminating simulation.")
            break

        # Vertex division
        if divide_vertex_mode == "random":
            chosen_vertex = random.choice(inside_vertices)
        elif divide_vertex_mode == "min":
            chosen_vertex = auxFunctionsHomeostasis.find_vertex_by_edge_sum(cellmap, vertex_list=inside_vertices, mode='min')
        elif divide_vertex_mode == "max":
            chosen_vertex = auxFunctionsHomeostasis.find_vertex_by_edge_sum(cellmap, vertex_list=inside_vertices, mode='max')
        else:
            raise ValueError("divide_vertex_mode must be 'random', 'min', or 'max'")

        print(f"Dividing vertex {chosen_vertex} (mode: {divide_vertex_mode})")

        # Executing vertex division
        cellmap, chosen_vertex, new_vertex_index, new_edge_index, opposite_edge_index = auxFunctionsHomeostasis.split_vertex_homeostasis(
            cellmap, chosen_vertex, geom, energyContributions_model, distance, retry_attempts=3
        )

        # Saving post-division visualisation
        if new_vertex_index is not None and new_vertex_index in cellmap.vert_df.index:
            fig, axes = auxFunctions.highlight_vertices(
                cellmap, geom, [new_vertex_index],
                default_color="white", highlight_color="red",
                default_size=5, highlight_size=100
            )
        else:
            fig, axes = auxFunctions.view(cellmap, geom)

        axes.set_xlim(initial_xlim)
        axes.set_ylim(initial_ylim)

        fig.savefig(
            os.path.join(output_folder, f"cycle_{cycle_index + 1}_division_frame.png"),
            dpi=150
        )
        plt.close(fig)

        # Allowing model to relax after division
        energyContributions_model.compute_energy(cellmap)
        cellmap, geom, _, _, _ = vertexModel2.solveEuler(
            cellmap, geom, energyContributions_model, endTime=40
        )

        auxFunctions.save_simulation_state(
            cellmap, output_folder, filename=f"cycle_{cycle_index + 1}_end_state.pkl"
        )

        print(f"Completed cycle {cycle_index + 1}")

    # Saving final state
    auxFunctions.save_simulation_state(cellmap, output_folder, filename="end_cellmap_state.pkl")
    print(f"Simulation finished after {cycle_index + 1} cycles.")

    return cellmap

In [ ]:
max_cycles = 500
max_layers = 1
distance = 0.1
collapse_edge_mode = "max"
divide_vertex_mode = "min"

output_folder = f'Homeostasis_simulation_{max_cycles}cycles_collapse_{collapse_edge_mode}_divide_{divide_vertex_mode}'


cellmap = homeostasis_simulation(
    output_folder = output_folder,
    max_cycles = max_cycles,
    max_layers = max_layers,
    distance = distance,
    collapse_edge_mode = collapse_edge_mode,
    divide_vertex_mode = divide_vertex_mode,
)